In [ ]:
%%writefile row_mapping_eval.py
"""
Row-mapping latency / energy evaluation of 32-bit MVM on 256x256 MAGIC crossbars:
HILO vs AUTO, normalized to AUTO, for the 15 SuiteSparse matrices.

Row layout (one 32x32 multiply per crossbar row):
  | ~A, ~B (inputs) | Partial product (reused) | Functional (reused) | Carry | Output |
Operands are written already complemented, so pp = NOR(~a, ~b) needs no NOT gates.

Per adder step:
  1. SET the partial-product segment                         1 cycle
  2. copy the carries it needs into the partial-product cells  2 NOTs per carry bit
  3. generate its new partial products                         1 NOR per bit
  4. run its netlist in the functional cells                   `cost` gates
     + 1 SET batch each time the functional cells run out
  5. lowest output bit -> Output (final); others -> Carry cells
One gate per cycle per row; all rows of a crossbar run the same gate in parallel.
"""
import math, re
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------ parameters ------------------------------
N = 32                 # operand width
ROW = 512             # cells per crossbar row
XB_ROWS = 512         # rows per crossbar
BUS_BITS = 256         # bits per cycle between crossbars (32 B bus)
ACC_BITS = 32          # width of the accumulated sum
ADD_STEPS = 322        # 32-bit add used in the reduction (same for both)
COPY_GATES = 2         # NOT + NOT per copied bit
# energy per operation (relative units; replace with device values, e.g. VTEAM)
E_NOR, E_SET, E_READ, E_WRITE, E_BUS = 1.0, 1.0, 1.0, 1.0, 1.0

BENCH = [("eris1176", 1176, 18552), ("cegb2919", 2919, 321543), ("raefsky1", 3242, 293409),
         ("fxm3_6", 5026, 94026), ("Na5", 5832, 305630), ("EX5", 6545, 295680),
         ("fp", 7548, 834222), ("ex40", 7740, 456188), ("benzene", 8219, 242669),
         ("bcsstk33", 8738, 591904), ("graham1", 9035, 335472), ("net25", 9520, 401200),
         ("bundle1", 10581, 770811), ("Si10H16", 17077, 875923), ("Goodwin_040", 17922, 561677)]

# ------------------------------ adder lists ------------------------------
HILO_TEXT = """
adder (2,3) W=4 cost=20 at column 1
adder (5,1,3) W=5 cost=42 at column 3
adder (6,4,4) W=5 cost=81 at column 4
adder (5,1,3,3) W=6 cost=60 at column 6
adder (10,8,8) W=6 cost=173 at column 7
adder (13) W=4 cost=79 at column 10
adder (15,3) W=5 cost=114 at column 11
adder (13) W=4 cost=79 at column 12
adder (17,7,3) W=6 cost=179 at column 13
adder (11,2) W=4 cost=77 at column 14
adder (15,3) W=5 cost=114 at column 15
adder (17,7) W=5 cost=158 at column 16
adder (15) W=4 cost=93 at column 17
adder (23,7,3) W=6 cost=230 at column 18
adder (17,7) W=5 cost=158 at column 19
adder (15) W=4 cost=93 at column 20
adder (25,4,7) W=6 cost=250 at column 21
adder (23,7,3) W=6 cost=230 at column 22
adder (15) W=4 cost=93 at column 23
adder (25,7) W=6 cost=219 at column 24
adder (23,7,4,1) W=6 cost=242 at column 25
adder (25,3) W=5 cost=191 at column 26
adder (25,7,3) W=6 cost=243 at column 27
adder (25,7,4) W=6 cost=249 at column 28
adder (25,7,4) W=6 cost=249 at column 29
adder (25,7,4,1) W=6 cost=255 at column 30
adder (25,7,4,1) W=6 cost=255 at column 31
adder (25,13,1,1) W=6 cost=281 at column 32
adder (17,7) W=5 cost=158 at column 33
adder (25,7,4,1) W=6 cost=255 at column 34
adder (25,3) W=5 cost=191 at column 35
adder (25,13,3) W=6 cost=288 at column 36
adder (17,9,7) W=6 cost=223 at column 37
adder (17,7,4,2) W=6 cost=200 at column 38
adder (15,8) W=5 cost=150 at column 39
adder (15) W=4 cost=93 at column 40
adder (25,1,1) W=5 cost=184 at column 41
adder (25,7,4,1) W=6 cost=255 at column 42
adder (17,7) W=5 cost=158 at column 43
adder (11,6,6,2) W=6 cost=159 at column 44
adder (15,8,8) W=6 cost=207 at column 45
adder (7,4) W=4 cost=61 at column 46
adder (7,4,4,4) W=6 cost=112 at column 47
adder (15,8,8) W=6 cost=207 at column 48
adder (7,4) W=4 cost=61 at column 49
adder (15,8,8) W=6 cost=207 at column 51
adder (7,4) W=4 cost=61 at column 52
adder (11,6,6,2) W=6 cost=159 at column 54
adder (5,1) W=3 cost=28 at column 55
adder (3,6,6,3) W=6 cost=104 at column 56
adder (3,4,3,1) W=5 cost=54 at column 59
"""

# AUTO: entry i = one single-column adder at column i (last vector entry = bits taken)
AUTO_HEIGHTS = [2, 4, 5, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 26, 27,
                28, 29, 30, 31, 32, 33, 34, 35, 36, 36, 35, 34, 33, 32, 31, 30, 29, 28, 27, 25, 24,
                23, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 12, 10, 9, 8, 7, 6, 5, 3, 2]
AUTO_WEIGHTS = [2, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6,
                6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 4, 4, 4, 4,
                4, 4, 3, 3, 3, 2, 2]
AUTO_COSTS = [5, 16, 22, 36, 45, 49, 58, 74, 79, 88, 93, 104, 109, 121, 126, 139, 148, 153, 166,
              173, 187, 192, 199, 203, 213, 220, 229, 233, 249, 254, 265, 265, 254, 249, 233, 229,
              220, 213, 203, 199, 192, 173, 166, 153, 148, 139, 134, 126, 121, 109, 104, 93, 88,
              79, 74, 58, 49, 45, 36, 31, 22, 11, 5]


def parse_hilo(txt):
    return [(list(map(int, m.group(1).split(","))), int(m.group(2)), int(m.group(3)), int(m.group(4)))
            for m in re.finditer(r"adder \(([\d,]+)\) W=(\d+) cost=(\d+) at column (\d+)", txt)]


AUTO = [([h], w, c, i + 1) for i, (h, w, c) in enumerate(zip(AUTO_HEIGHTS, AUTO_WEIGHTS, AUTO_COSTS))]
HILO = parse_hilo(HILO_TEXT)

# partial-product bits per column (bit 0 = a0b0 needs no adder): 2, 3, ..., 32, 31, ..., 1
PP_INIT = {c: (c + 1 if c <= N - 1 else 2 * N - 1 - c) for c in range(1, 2 * N - 1)}


# ------------------------------ one multiplication ------------------------------
def schedule(G):
    """Column sweep: which input bits of each adder are fresh partial products and
    which are carries copied from earlier adders; peak number of waiting carries."""
    pp_left = defaultdict(int, PP_INIT)
    waiting = defaultdict(int)
    steps, peak = [], 0
    for k, w, cost, col in G:
        copied = new_pp = 0
        for j, kj in enumerate(k):
            take = min(waiting[col + j], kj)
            waiting[col + j] -= take
            copied += take
            new_pp += kj - take
            pp_left[col + j] -= kj - take
            if pp_left[col + j] < 0:
                raise ValueError(f"adder at column {col} takes more bits than exist")
        for j in range(w):
            waiting[col + j] += 1
        # a column with nothing left to add holds its final bit -> Output segment
        for c in list(waiting):
            if c <= col and pp_left[c] == 0 and waiting[c] == 1:
                waiting[c] = 0
        peak = max(peak, sum(waiting.values()))
        steps.append(dict(inputs=sum(k), copied=copied, new_pp=new_pp, W=w, cost=cost))
    if any(v for v in pp_left.values()):
        raise ValueError("some partial products are never added")
    return steps, peak


def multiply(G):
    steps, carry_peak = schedule(G)
    seg = dict(inputs=2 * N, partial_product=max(s["inputs"] for s in steps),
               carry=carry_peak, output=2 * N)
    fixed = sum(seg.values())
    F = ROW - fixed
    if F <= 0:
        raise ValueError("multiplication does not fit in the row")
    ops = defaultdict(int)
    for s in steps:
        ops["SET_pp"] += 1
        ops["NOT_copy"] += COPY_GATES * s["copied"]
        ops["NOR_pp"] += s["new_pp"]
        ops["gates"] += s["cost"]
        ops["SET_func"] += max(1, math.ceil(s["cost"] / F))
        ops["carries"] += s["copied"]
    evals = ops["NOT_copy"] + ops["NOR_pp"] + ops["gates"]
    cycles = evals + ops["SET_pp"] + ops["SET_func"] + 1        # +1 initial SET of the row
    energy = evals * (E_NOR + E_SET)                            # every output cell SET once
    return dict(steps=len(steps), seg=seg, fixed=fixed, functional=F, ops=dict(ops),
                cycles=cycles, energy=energy)


# ------------------------------ MVM on the crossbars ------------------------------
def bus(bits):
    return (2 + math.ceil(bits / BUS_BITS)) if bits else 0     # read + transfer + write


def mvm(n, nnz, mul):
    """y = A x, one nonzero per crossbar row; all crossbars work in parallel."""
    q, r = divmod(nnz, n)
    lens = [q + 1] * r + [q] * (n - r)                          # nnz spread over the rows
    t, e = {}, {}
    t["load"] = bus(n * N) + XB_ROWS
    e["load"] = n * N * (E_READ + E_BUS + E_WRITE) + nnz * N * E_WRITE
    t["multiply"] = mul["cycles"]
    e["multiply"] = nnz * mul["energy"]
    # pack matrix rows into crossbars, then a pairwise tree per crossbar
    xbars, cur, fill, split = [], [], 0, 0
    for L in lens:
        while L:
            take = min(L, XB_ROWS - fill)
            cur.append(take); fill += take; L -= take
            if L:
                split += 1
            if fill == XB_ROWS:
                xbars.append(cur); cur, fill = [], 0
    if cur:
        xbars.append(cur)
    copy_cyc, worst, pairs_all = 2 * (COPY_GATES), 0, 0     # column-parallel copy of 32 bits
    for segs in xbars:
        lv = max((math.ceil(math.log2(s)) if s > 1 else 0) for s in segs)
        pairs = sum(sum(math.ceil(s / 2 ** l) // 2 for s in segs) for l in range(lv))
        worst = max(worst, pairs * copy_cyc + lv * ADD_STEPS)
        pairs_all += pairs
    t["reduce"] = worst
    e["reduce"] = (pairs_all * ACC_BITS * COPY_GATES * (E_NOR + E_SET)
                   + (nnz - n) * ADD_STEPS * (E_NOR + E_SET))
    t["merge"] = (bus(split * ACC_BITS) + ADD_STEPS) if split else 0
    e["merge"] = split * ACC_BITS * (E_READ + E_BUS + E_WRITE) + split * ADD_STEPS * (E_NOR + E_SET)
    t["readout"] = XB_ROWS + bus(n * ACC_BITS)
    e["readout"] = n * ACC_BITS * (E_READ + E_BUS + E_WRITE)
    return sum(t.values()), sum(e.values()), len(xbars), t


# ------------------------------ run ------------------------------
MUL = {"AUTO": multiply(AUTO), "HILO": multiply(HILO)}

print(f"Per 32x32 multiplication, {ROW}-cell row")
print(f"{'':6s}{'steps':>6s}{'gates':>7s}{'carries':>8s}{'pp NOR':>7s}{'SET':>6s}{'cycles':>8s}"
      f"{'inputs':>7s}{'pp seg':>7s}{'carry':>6s}{'output':>7s}{'fixed':>6s}{'func':>5s}")
for t, m in MUL.items():
    o, s = m["ops"], m["seg"]
    print(f"{t:6s}{m['steps']:6d}{o['gates']:7d}{o['carries']:8d}{o['NOR_pp']:7d}"
          f"{o['SET_pp'] + o['SET_func']:6d}{m['cycles']:8d}{s['inputs']:7d}{s['partial_product']:7d}"
          f"{s['carry']:6d}{s['output']:7d}{m['fixed']:6d}{m['functional']:5d}")
print("cells per multiplication = fixed + functional =", ROW, "(one multiply per row)")

names = [b[0] for b in BENCH] + ["GMean"]
lat, en = {"AUTO": [], "HILO": []}, {"AUTO": [], "HILO": []}
print(f"\n{'benchmark':12s}{'xbars':>7s}{'AUTO cyc':>10s}{'HILO cyc':>10s}{'norm L':>8s}{'norm E':>8s}")
for name, n, nnz in BENCH:
    la, ea, xb, _ = mvm(n, nnz, MUL["AUTO"])
    lh, eh, _, _ = mvm(n, nnz, MUL["HILO"])
    lat["AUTO"].append(1.0); en["AUTO"].append(1.0)
    lat["HILO"].append(lh / la); en["HILO"].append(eh / ea)
    print(f"{name:12s}{xb:7d}{la:10d}{lh:10d}{lh / la:8.3f}{eh / ea:8.3f}")
gm = lambda v: float(np.exp(np.mean(np.log(v))))
for t in lat:
    lat[t].append(gm(lat[t])); en[t].append(gm(en[t]))
print(f"{'GMean':12s}{'':27s}{lat['HILO'][-1]:8.3f}{en['HILO'][-1]:8.3f}")

# ------------------------------ figure ------------------------------
x = np.arange(len(names), dtype=float); x[-1] += 0.6
w = 0.36
style = {"AUTO": ("#1D9E75", "xxx"), "HILO": ("#EF9F27", "///")}
fig, axs = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for ax, data, lab in [(axs[0], lat, "Norm. latency"), (axs[1], en, "Norm. energy")]:
    for j, t in enumerate(data):
        col, hat = style[t]
        ax.bar(x + (j - 0.5) * w, data[t], w, label=t, color=col, hatch=hat,
               edgecolor="black", linewidth=0.4)
    ax.axhline(1.0, color="black", linewidth=0.6, linestyle="--")
    ax.set_ylabel(lab, fontweight="bold"); ax.set_ylim(0, 1.15); ax.grid(axis="y", alpha=0.3)
axs[1].set_xticks(x); axs[1].set_xticklabels(names, rotation=40, ha="right")
axs[1].get_xticklabels()[-1].set_fontweight("bold")
axs[0].legend(ncol=2, loc="upper center", bbox_to_anchor=(0.5, 1.22), frameon=False)
fig.tight_layout()
fig.savefig("mvm_hilo_vs_auto_256.png", dpi=300)
fig.savefig("mvm_hilo_vs_auto_256.pdf")
plt.show()


In [ ]:
%%writefile reduction_eval.py
"""
Step 2 of each dot product (adding its products) done with each technique's own adders.

Adding k numbers of 32 bits = a flat bit block with k bits in every column. The same
column-sweep search that found the multiplication cover finds the cheapest cover of it:
  HILO: full adder library        AUTO-style: single-column adders only
The reduction of every matrix row then uses these costs instead of one fixed 322-step adder.
Run after row_mapping_eval.py (uses MUL, BENCH, N, bus, ... from it).
"""
import csv, heapq, math, os, time
from fractions import Fraction

LOOKUP = ("lookuptable_with_weights.csv" if os.path.exists("lookuptable_with_weights.csv")
          else "../lookuptable_with_weights.csv")   # repository root
KS = [2, 3, 4]              # how many numbers one adder-block adds at once
WIDTH = 32                  # accumulator width


def load_lib(path, single_column=False):
    lib = []
    with open(path, newline="") as f:
        for r in csv.DictReader(f):
            v = [int(r[f"col{i}"]) for i in range(1, 9)]
            while v and v[-1] == 0:
                v.pop()
            if single_column and len(v) != 1:
                continue
            lib.append((tuple(v), int(r["weight"]), int(round(float(r["Cost"])))))
    lib.sort(key=lambda a: a[0][0])
    return lib


def add_cover(lib, k, width=WIDTH, goal="relaxed"):
    """Cheapest cover of k numbers of `width` bits (column-sweep A*, as in hilo_sweep.py)."""
    C = width + 1 + (math.ceil(math.log2(k)) if k > 1 else 0)
    init = [k if c < width else 0 for c in range(C + 8)]
    suffix = [0] * (C + 9)
    for c in range(C + 7, -1, -1):
        suffix[c] = suffix[c + 1] + init[c]
    rho = min(Fraction(c, sum(v) - w) for v, w, c in lib if sum(v) - w > 0)
    col = lambda k0, win, c: win[c - k0] if c - k0 < 6 else init[c]

    def h(k0, win):
        if k0 >= C:
            return 0
        ex = sum(win) + suffix[k0 + 6] - (C - k0)
        return 0 if ex <= 0 else -((-ex * rho.numerator) // rho.denominator)

    k0 = next(c for c in range(C) if init[c] >= 2)
    start = (k0, tuple(init[k0:k0 + 6]))
    g, parent, heap = {start: 0}, {start: None}, [(h(*start), 0, start)]
    while heap:
        _, gs, st = heapq.heappop(heap)
        if gs > g[st]:
            continue
        kk, win = st
        if kk >= C:
            steps = 0
            while parent[st]:
                st = parent[st][0]; steps += 1
            return gs, steps
        for v, w, cost in lib:
            if v[0] > win[0]:
                break
            if kk + w - 1 >= C or any(v[j] > win[j] for j in range(1, len(v))):
                continue
            t = [win[j] - (v[j] if j < len(v) else 0) + (1 if j < w else 0) for j in range(6)]
            nk, dead = kk, False
            while nk < C and col(kk, t, nk) < 2:
                if goal == "paper" and col(kk, t, nk) == 0:
                    dead = True
                    break
                nk += 1
            if dead:
                continue
            nwin = tuple(col(kk, t, c) if c < C else 0 for c in range(nk, nk + 6))
            ns, ng = (nk, nwin), gs + cost
            if ng < g.get(ns, 1 << 60):
                g[ns], parent[ns] = ng, (st, None)
                heapq.heappush(heap, (ng + h(nk, nwin), ng, ns))
    return None, 0


# ---------------------- cost of adding k numbers, per technique ----------------------
LIBS = {"HILO": load_lib(LOOKUP), "AUTO": load_lib(LOOKUP, single_column=True)}
ADD = {t: {} for t in LIBS}
print(f"Cost of adding k numbers of {WIDTH} bits (gates; cycles = gates + 1 SET per adder)")
for t, lib in LIBS.items():
    for k in KS:
        t0 = time.time()
        gates, steps = add_cover(lib, k)
        ADD[t][k] = dict(gates=gates, cycles=gates + steps)
        print(f"  {t}: k={k}  gates={gates}  adders={steps}  ({time.time() - t0:.1f}s)")


# ---------------------- MVM with technique-specific reduction ----------------------
def reduce_cost(segments, add):
    """k-ary tree inside one crossbar. Per level: copy the partial sums of each group into
    one row (2 NOTs, all 32 bits in parallel = 4 cycles per copied row), then one k-input add.
    Returns (cycles, gate evaluations, row copies)."""
    best = None
    for k in KS:
        cyc = ev = cp = 0
        level = list(segments)
        while max(level) > 1:
            # rows of this crossbar work on their own groups in parallel; copies are sequential
            copies = sum(s - math.ceil(s / k) for s in level)
            sizes = [min(k, s) for s in level if s > 1]
            cyc += copies * 2 * COPY_GATES + max(add[sz]["cycles"] for sz in sizes)
            ev += sum((s // k) * add[k]["gates"] + (add[s % k]["gates"] if s % k > 1 else 0)
                      for s in level)
            cp += copies
            level = [math.ceil(s / k) for s in level]
        if best is None or cyc < best[0]:
            best = (cyc, ev, cp)
    return best


def mvm2(n, nnz, mul, add):
    q, r = divmod(nnz, n)
    lens = [q + 1] * r + [q] * (n - r)
    xbars, cur, fill, split = [], [], 0, 0
    for L in lens:
        while L:
            take = min(L, XB_ROWS - fill)
            cur.append(take); fill += take; L -= take
            if L:
                split += 1
            if fill == XB_ROWS:
                xbars.append(cur); cur, fill = [], 0
    if cur:
        xbars.append(cur)
    t = {"load": bus(n * N) + XB_ROWS, "multiply": mul["cycles"]}
    e = {"load": n * N * (E_READ + E_BUS + E_WRITE) + nnz * N * E_WRITE,
         "multiply": nnz * mul["energy"]}
    worst, ev, cp = 0, 0, 0
    for segs in xbars:
        c, g_, p = reduce_cost(segs, add)
        worst = max(worst, c); ev += g_; cp += p
    t["reduce"] = worst
    e["reduce"] = ev * (E_NOR + E_SET) + cp * ACC_BITS * COPY_GATES * (E_NOR + E_SET)
    t["merge"] = (bus(split * ACC_BITS) + add[2]["cycles"]) if split else 0
    e["merge"] = split * (ACC_BITS * (E_READ + E_BUS + E_WRITE) + add[2]["gates"] * (E_NOR + E_SET))
    t["readout"] = XB_ROWS + bus(n * ACC_BITS)
    e["readout"] = n * ACC_BITS * (E_READ + E_BUS + E_WRITE)
    return sum(t.values()), sum(e.values()), t


names2 = [b[0] for b in BENCH] + ["GMean"]
L2, E2 = {"AUTO": [], "HILO": []}, {"AUTO": [], "HILO": []}
print(f"\n{'benchmark':12s}{'nnz/row':>8s}{'AUTO cyc':>10s}{'HILO cyc':>10s}"
      f"{'reduce share':>13s}{'norm L':>8s}{'norm E':>8s}")
for name, n, nnz in BENCH:
    la, ea, ta = mvm2(n, nnz, MUL["AUTO"], ADD["AUTO"])
    lh, eh, _ = mvm2(n, nnz, MUL["HILO"], ADD["HILO"])
    L2["AUTO"].append(1.0); E2["AUTO"].append(1.0)
    L2["HILO"].append(lh / la); E2["HILO"].append(eh / ea)
    print(f"{name:12s}{nnz / n:8.1f}{la:10d}{lh:10d}{ta['reduce'] / la:13.2f}"
          f"{lh / la:8.3f}{eh / ea:8.3f}")
for t in L2:
    L2[t].append(gm(L2[t])); E2[t].append(gm(E2[t]))
print(f"{'GMean':12s}{'':41s}{L2['HILO'][-1]:8.3f}{E2['HILO'][-1]:8.3f}")

x2 = np.arange(len(names2), dtype=float); x2[-1] += 0.6
fig, axs = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for ax, data, lab in [(axs[0], L2, "Norm. latency"), (axs[1], E2, "Norm. energy")]:
    for j, t in enumerate(data):
        colr, hat = style[t]
        ax.bar(x2 + (j - 0.5) * w, data[t], w, label=t, color=colr, hatch=hat,
               edgecolor="black", linewidth=0.4)
    ax.axhline(1.0, color="black", linewidth=0.6, linestyle="--")
    ax.set_ylabel(lab, fontweight="bold"); ax.set_ylim(0, 1.15); ax.grid(axis="y", alpha=0.3)
axs[1].set_xticks(x2); axs[1].set_xticklabels(names2, rotation=40, ha="right")
axs[1].get_xticklabels()[-1].set_fontweight("bold")
axs[0].legend(ncol=2, loc="upper center", bbox_to_anchor=(0.5, 1.22), frameon=False)
fig.tight_layout()
fig.savefig("mvm_hilo_vs_auto_256_own_reduction.png", dpi=300)
fig.savefig("mvm_hilo_vs_auto_256_own_reduction.pdf")
plt.show()


In [ ]:
# Multiply evaluation first (defines MUL, BENCH, ...), then the reduction with each technique's own adders
import os; print('lookup table found:', any(os.path.exists(p) for p in ('lookuptable_with_weights.csv', '../lookuptable_with_weights.csv')))
%run -i row_mapping_eval.py
%run -i reduction_eval.py

In [ ]:
# Redraw the HILO vs AUTO chart (own-adder reduction) with the y-axis starting at 0.9
import numpy as np, matplotlib.pyplot as plt
YMIN, YMAX = 0.90, 1.01
x2 = np.arange(len(names2), dtype=float); x2[-1] += 0.6
w2 = 0.36
style2 = {"AUTO": ("#1D9E75", "xxx"), "HILO": ("#EF9F27", "///")}
fig, axs = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for ax, data, lab in [(axs[0], L2, "Norm. latency"), (axs[1], E2, "Norm. energy")]:
    for j, t in enumerate(data):
        colr, hat = style2[t]
        bars = ax.bar(x2 + (j - 0.5) * w2, data[t], w2, label=t, color=colr, hatch=hat,
                      edgecolor="black", linewidth=0.4)
        if t == "HILO":
            for b, v in zip(bars, data[t]):
                ax.text(b.get_x() + b.get_width() / 2, v - 0.003, f"{v:.3f}",
                        ha="center", va="top", fontsize=7, rotation=90)
    ax.axhline(1.0, color="black", linewidth=0.6, linestyle="--")
    ax.set_ylim(YMIN, YMAX)
    ax.set_yticks(np.arange(YMIN, 1.0001, 0.02))
    ax.set_ylabel(lab, fontweight="bold"); ax.grid(axis="y", alpha=0.3)
axs[1].set_xticks(x2); axs[1].set_xticklabels(names2, rotation=40, ha="right")
axs[1].get_xticklabels()[-1].set_fontweight("bold")
axs[0].legend(ncol=2, loc="upper center", bbox_to_anchor=(0.5, 1.22), frameon=False)
fig.tight_layout()
fig.savefig("mvm_hilo_vs_auto_256_own_reduction_zoom.png", dpi=300)
fig.savefig("mvm_hilo_vs_auto_256_own_reduction_zoom.pdf")
plt.show()


In [ ]:
# Energy = component power x active time per phase (as in AUTO / LOGIC), per crossbar
# Powers from AUTO Table III (mW); energy in pJ because mW x ns = pJ
import math, numpy as np, matplotlib.pyplot as plt
T_CYCLE_NS = 1.0                      # one MAGIC step (cancels in the normalized plot)
P = dict(xbar=1.20, driver=0.65, sense_amp=0.58, local_bus=4.66, global_bus=26.0)

# components that are ON in each phase (per crossbar), and whether the global bus is used
ACTIVE = {
    "load":     (["driver", "sense_amp", "local_bus"], True),   # x in over the bus, written to rows
    "multiply": (["xbar", "driver"], False),                    # MAGIC gates in every row
    "reduce":   (["xbar", "driver"], False),                    # row copies + adds, in-crossbar
    "merge":    (["sense_amp", "local_bus"], True),             # partial sums between crossbars
    "readout":  (["sense_amp", "local_bus"], True),             # y out over the bus
}


def phase_energy(t, n_xbars):
    e = {}
    for ph, cyc in t.items():
        comps, uses_bus = ACTIVE[ph]
        p = n_xbars * sum(P[c] for c in comps) + (P["global_bus"] if uses_bus else 0.0)
        e[ph] = p * cyc * T_CYCLE_NS          # pJ
    return e


LP, EP = {"AUTO": [], "HILO": []}, {"AUTO": [], "HILO": []}
print(f"{'benchmark':12s}{'xbars':>7s}{'AUTO uJ':>10s}{'HILO uJ':>10s}{'norm L':>8s}{'norm E':>8s}")
for name, n, nnz in BENCH:
    xb = math.ceil(nnz / XB_ROWS)
    la, _, ta = mvm2(n, nnz, MUL["AUTO"], ADD["AUTO"])
    lh, _, th = mvm2(n, nnz, MUL["HILO"], ADD["HILO"])
    ea = sum(phase_energy(ta, xb).values()); eh = sum(phase_energy(th, xb).values())
    LP["AUTO"].append(1.0); EP["AUTO"].append(1.0)
    LP["HILO"].append(lh / la); EP["HILO"].append(eh / ea)
    print(f"{name:12s}{xb:7d}{ea / 1e6:10.2f}{eh / 1e6:10.2f}{lh / la:8.3f}{eh / ea:8.3f}")
for t in LP:
    LP[t].append(gm(LP[t])); EP[t].append(gm(EP[t]))
print(f"{'GMean':12s}{'':27s}{LP['HILO'][-1]:8.3f}{EP['HILO'][-1]:8.3f}")

YMIN, YMAX = 0.90, 1.01
x3 = np.arange(len(names2), dtype=float); x3[-1] += 0.6
w3 = 0.36
st3 = {"AUTO": ("#1D9E75", "xxx"), "HILO": ("#EF9F27", "///")}
fig, axs = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for ax, data, lab in [(axs[0], LP, "Norm. latency"), (axs[1], EP, "Norm. energy")]:
    for j, t in enumerate(data):
        colr, hat = st3[t]
        ax.bar(x3 + (j - 0.5) * w3, data[t], w3, label=t, color=colr, hatch=hat,
               edgecolor="black", linewidth=0.4)
    ax.axhline(1.0, color="black", linewidth=0.6, linestyle="--")
    ax.set_ylim(YMIN, YMAX); ax.set_yticks(np.arange(YMIN, 1.0001, 0.02))
    ax.set_ylabel(lab, fontweight="bold"); ax.grid(axis="y", alpha=0.3)
axs[1].set_xticks(x3); axs[1].set_xticklabels(names2, rotation=40, ha="right")
axs[1].get_xticklabels()[-1].set_fontweight("bold")
axs[0].legend(ncol=2, loc="upper center", bbox_to_anchor=(0.5, 1.22), frameon=False)
fig.tight_layout()
fig.savefig("mvm_hilo_vs_auto_256_power_energy.png", dpi=300)
fig.savefig("mvm_hilo_vs_auto_256_power_energy.pdf")
plt.show()


In [ ]:
# Partial-product generation = 3n^2 (NOT a, NOT b, NOR per bit), the same for AUTO and HILO.
# Reuses the addition costs ADD already found (no new search) and the power model above.
import math, numpy as np, matplotlib.pyplot as plt
from collections import defaultdict
PP_OPS = 3                      # operations per partial product (3 -> 3n^2 in total)
print(f"array: {ROW}-cell rows, {XB_ROWS} rows per crossbar | partial products: {PP_OPS} ops each")


def multiply_pp(G, pp_ops=PP_OPS):
    steps, carry_peak = schedule(G)
    seg = dict(inputs=2 * N, partial_product=max(s["inputs"] for s in steps),
               carry=carry_peak, output=2 * N)
    F = ROW - sum(seg.values()) - (2 if pp_ops == 3 else 0)   # 2 cells hold ~a, ~b while a pp is made
    ops = defaultdict(int)
    for s in steps:
        ops["SET_pp"] += 1
        ops["NOT_copy"] += COPY_GATES * s["copied"]
        ops["pp_gen"] += pp_ops * s["new_pp"]
        ops["gates"] += s["cost"]
        ops["SET_func"] += max(1, math.ceil(s["cost"] / F))
    ops["pp_gen"] += pp_ops                                     # a0*b0 (final bit 0)
    evals = ops["NOT_copy"] + ops["pp_gen"] + ops["gates"]
    cycles = evals + ops["SET_pp"] + ops["SET_func"] + 1
    return dict(cycles=cycles, energy=evals * (E_NOR + E_SET), functional=F, ops=dict(ops))


MUL3 = {"AUTO": multiply_pp(AUTO), "HILO": multiply_pp(HILO)}
print(f"\n{'':6s}{'adder gates':>12s}{'pp ops':>8s}{'copies':>8s}{'SET':>6s}{'cycles':>8s}{'func':>6s}")
for t, m in MUL3.items():
    o = m["ops"]
    print(f"{t:6s}{o['gates']:12d}{o['pp_gen']:8d}{o['NOT_copy']:8d}"
          f"{o['SET_pp'] + o['SET_func']:6d}{m['cycles']:8d}{m['functional']:6d}")
print(f"HILO saving per multiply: {100 * (1 - MUL3['HILO']['cycles'] / MUL3['AUTO']['cycles']):.1f}%")

L3, EP3, EO3 = ({"AUTO": [], "HILO": []} for _ in range(3))
print(f"\n{'benchmark':12s}{'norm L':>8s}{'norm E (power x time)':>23s}{'norm E (per op)':>17s}")
for name, n, nnz in BENCH:
    xb = math.ceil(nnz / XB_ROWS)
    la, eoa, ta = mvm2(n, nnz, MUL3["AUTO"], ADD["AUTO"])
    lh, eoh, th = mvm2(n, nnz, MUL3["HILO"], ADD["HILO"])
    epa = sum(phase_energy(ta, xb).values()); eph = sum(phase_energy(th, xb).values())
    for d, v in ((L3, lh / la), (EP3, eph / epa), (EO3, eoh / eoa)):
        d["AUTO"].append(1.0); d["HILO"].append(v)
    print(f"{name:12s}{lh / la:8.3f}{eph / epa:23.3f}{eoh / eoa:17.3f}")
for d in (L3, EP3, EO3):
    for t in d:
        d[t].append(gm(d[t]))
print(f"{'GMean':12s}{L3['HILO'][-1]:8.3f}{EP3['HILO'][-1]:23.3f}{EO3['HILO'][-1]:17.3f}")

YMIN, YMAX = 0.90, 1.01
x4 = np.arange(len(names2), dtype=float); x4[-1] += 0.6
w4 = 0.36
st4 = {"AUTO": ("#1D9E75", "xxx"), "HILO": ("#EF9F27", "///")}
fig, axs = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for ax, data, lab in [(axs[0], L3, "Norm. latency"), (axs[1], EP3, "Norm. energy")]:
    for j, t in enumerate(data):
        colr, hat = st4[t]
        ax.bar(x4 + (j - 0.5) * w4, data[t], w4, label=t, color=colr, hatch=hat,
               edgecolor="black", linewidth=0.4)
    ax.axhline(1.0, color="black", linewidth=0.6, linestyle="--")
    ax.set_ylim(YMIN, YMAX); ax.set_yticks(np.arange(YMIN, 1.0001, 0.02))
    ax.set_ylabel(lab, fontweight="bold"); ax.grid(axis="y", alpha=0.3)
axs[1].set_xticks(x4); axs[1].set_xticklabels(names2, rotation=40, ha="right")
axs[1].get_xticklabels()[-1].set_fontweight("bold")
axs[0].legend(ncol=2, loc="upper center", bbox_to_anchor=(0.5, 1.22), frameon=False)
fig.tight_layout()
tag = f"{ROW}x{XB_ROWS}_pp3n2"
fig.savefig(f"mvm_hilo_vs_auto_{tag}.png", dpi=300)
fig.savefig(f"mvm_hilo_vs_auto_{tag}.pdf")
plt.show()


In [ ]:
# Actual latency and energy of AUTO and HILO per matrix, and HILO's improvement
# (percentage and "times"). Uses the 3n^2 multiply (MUL3), own-adder reduction (ADD) and the
# power x time energy model above. Latency = cycles x T_CYCLE_NS (set your MAGIC cycle time).
import math, numpy as np, pandas as pd

T_CYCLE_NS = 1.0          # ns per MAGIC cycle  <-- set from your device model
rows = []
for name, n, nnz in BENCH:
    xb = math.ceil(nnz / XB_ROWS)
    ca, _, ta = mvm2(n, nnz, MUL3["AUTO"], ADD["AUTO"])
    ch, _, th = mvm2(n, nnz, MUL3["HILO"], ADD["HILO"])
    ea = sum(phase_energy(ta, xb).values()) * T_CYCLE_NS / 1e6      # pJ -> uJ
    eh = sum(phase_energy(th, xb).values()) * T_CYCLE_NS / 1e6
    la, lh = ca * T_CYCLE_NS / 1e3, ch * T_CYCLE_NS / 1e3            # ns -> us
    rows.append({"Matrix": name, "nnz": nnz,
                 "AUTO latency (us)": la, "HILO latency (us)": lh,
                 "Latency saving (%)": 100 * (1 - lh / la), "Speed-up (x)": la / lh,
                 "AUTO energy (uJ)": ea, "HILO energy (uJ)": eh,
                 "Energy saving (%)": 100 * (1 - eh / ea), "Energy ratio (x)": ea / eh})

df = pd.DataFrame(rows)
g = lambda s: float(np.exp(np.mean(np.log(s))))
sp, er = g(df["Speed-up (x)"]), g(df["Energy ratio (x)"])
df.loc[len(df)] = {"Matrix": "GMean", "nnz": np.nan,
                   "AUTO latency (us)": g(df["AUTO latency (us)"]),
                   "HILO latency (us)": g(df["HILO latency (us)"]),
                   "Latency saving (%)": 100 * (1 - 1 / sp), "Speed-up (x)": sp,
                   "AUTO energy (uJ)": g(df["AUTO energy (uJ)"]),
                   "HILO energy (uJ)": g(df["HILO energy (uJ)"]),
                   "Energy saving (%)": 100 * (1 - 1 / er), "Energy ratio (x)": er}

print(f"Array {ROW}x{XB_ROWS}, partial products {PP_OPS} ops/bit, cycle time {T_CYCLE_NS} ns")
fmt = {"nnz": "{:,.0f}", "AUTO latency (us)": "{:.2f}", "HILO latency (us)": "{:.2f}",
       "Latency saving (%)": "{:.2f}", "Speed-up (x)": "{:.4f}",
       "AUTO energy (uJ)": "{:.2f}", "HILO energy (uJ)": "{:.2f}",
       "Energy saving (%)": "{:.2f}", "Energy ratio (x)": "{:.4f}"}
display(df.style.format(fmt, na_rep="").hide(axis="index"))
df.to_csv(f"hilo_vs_auto_absolute_{ROW}x{XB_ROWS}.csv", index=False)
print(f"saved hilo_vs_auto_absolute_{ROW}x{XB_ROWS}.csv")


In [ ]:
# Overall HILO vs AUTO: total over all 15 matrices (one MVM each), plus the geometric mean
import pandas as pd
per = df[df["Matrix"] != "GMean"]
tot = {k: per[k].sum() for k in ["AUTO latency (us)", "HILO latency (us)", "AUTO energy (uJ)", "HILO energy (uJ)"]}
gmr = df[df["Matrix"] == "GMean"].iloc[0]

summary = pd.DataFrame([
    {"Metric": "Latency, total of 15 MVMs (us)", "AUTO": tot["AUTO latency (us)"], "HILO": tot["HILO latency (us)"]},
    {"Metric": "Energy, total of 15 MVMs (uJ)",  "AUTO": tot["AUTO energy (uJ)"],  "HILO": tot["HILO energy (uJ)"]},
    {"Metric": "Latency, geometric mean (us)",   "AUTO": gmr["AUTO latency (us)"], "HILO": gmr["HILO latency (us)"]},
    {"Metric": "Energy, geometric mean (uJ)",    "AUTO": gmr["AUTO energy (uJ)"],  "HILO": gmr["HILO energy (uJ)"]},
])
summary["Saving (%)"] = 100 * (1 - summary["HILO"] / summary["AUTO"])
summary["Improvement (x)"] = summary["AUTO"] / summary["HILO"]

print(f"HILO vs AUTO overall | array {ROW}x{XB_ROWS}, partial products {PP_OPS} ops/bit, cycle {T_CYCLE_NS} ns")
display(summary.style.format({"AUTO": "{:.2f}", "HILO": "{:.2f}", "Saving (%)": "{:.2f}",
                              "Improvement (x)": "{:.4f}"}).hide(axis="index"))
summary.to_csv(f"hilo_vs_auto_overall_{ROW}x{XB_ROWS}.csv", index=False)


In [ ]:
# Overall HILO vs AUTO with and without partial-product generation (pp ops/bit = 3 -> 3n^2, 0 -> none)
import math, numpy as np, pandas as pd


def overall(pp_ops):
    mul = {"AUTO": multiply_pp(AUTO, pp_ops), "HILO": multiply_pp(HILO, pp_ops)}
    L = {"AUTO": [], "HILO": []}; E = {"AUTO": [], "HILO": []}
    for name, n, nnz in BENCH:
        xb = math.ceil(nnz / XB_ROWS)
        for t in mul:
            c, _, ph = mvm2(n, nnz, mul[t], ADD[t])
            L[t].append(c * T_CYCLE_NS / 1e3)                                   # us
            E[t].append(sum(phase_energy(ph, xb).values()) * T_CYCLE_NS / 1e6)  # uJ
    gmean = lambda v: float(np.exp(np.mean(np.log(v))))
    rows = [("Multiply, cycles per product", mul["AUTO"]["cycles"], mul["HILO"]["cycles"]),
            ("Latency, total of 15 MVMs (us)", sum(L["AUTO"]), sum(L["HILO"])),
            ("Energy, total of 15 MVMs (uJ)", sum(E["AUTO"]), sum(E["HILO"])),
            ("Latency, geometric mean (us)", gmean(L["AUTO"]), gmean(L["HILO"])),
            ("Energy, geometric mean (uJ)", gmean(E["AUTO"]), gmean(E["HILO"]))]
    return pd.DataFrame([{"PP generation": "3n^2" if pp_ops == 3 else ("none" if pp_ops == 0 else f"{pp_ops} ops/bit"),
                          "Metric": m, "AUTO": a, "HILO": h,
                          "Saving (%)": 100 * (1 - h / a), "Improvement (x)": a / h} for m, a, h in rows])


both = pd.concat([overall(3), overall(0)], ignore_index=True)
print(f"HILO vs AUTO overall | array {ROW}x{XB_ROWS}, cycle {T_CYCLE_NS} ns")
display(both.style.format({"AUTO": "{:,.2f}", "HILO": "{:,.2f}", "Saving (%)": "{:.2f}",
                           "Improvement (x)": "{:.4f}"}).hide(axis="index"))
both.to_csv(f"hilo_vs_auto_overall_pp3_vs_none_{ROW}x{XB_ROWS}.csv", index=False)


In [ ]:
# HILO vs AUTO WITHOUT partial-product generation (adders, carry copies, SETs, reduction only)
# Bar chart per matrix (normalized to AUTO) + overall improvement (total and geometric mean)
import math, numpy as np, pandas as pd, matplotlib.pyplot as plt

MUL0 = {"AUTO": multiply_pp(AUTO, 0), "HILO": multiply_pp(HILO, 0)}
print(f"Array {ROW}x{XB_ROWS}, no partial-product generation, cycle {T_CYCLE_NS} ns")
print(f"Multiply cycles per product: AUTO {MUL0['AUTO']['cycles']}, HILO {MUL0['HILO']['cycles']} "
      f"({100 * (1 - MUL0['HILO']['cycles'] / MUL0['AUTO']['cycles']):.2f}% saving)")

lat0 = {"AUTO": [], "HILO": []}; en0 = {"AUTO": [], "HILO": []}
for name, n, nnz in BENCH:
    xb = math.ceil(nnz / XB_ROWS)
    for t in MUL0:
        c, _, ph = mvm2(n, nnz, MUL0[t], ADD[t])
        lat0[t].append(c * T_CYCLE_NS / 1e3)                                   # us
        en0[t].append(sum(phase_energy(ph, xb).values()) * T_CYCLE_NS / 1e6)  # uJ

gmean = lambda v: float(np.exp(np.mean(np.log(v))))
nl = [h / a for h, a in zip(lat0["HILO"], lat0["AUTO"])]
ne = [h / a for h, a in zip(en0["HILO"], en0["AUTO"])]

# ---------------- overall improvement ----------------
rows = [("Latency, total of 15 MVMs (us)", sum(lat0["AUTO"]), sum(lat0["HILO"])),
        ("Energy, total of 15 MVMs (uJ)", sum(en0["AUTO"]), sum(en0["HILO"])),
        ("Latency, geometric mean (us)", gmean(lat0["AUTO"]), gmean(lat0["HILO"])),
        ("Energy, geometric mean (uJ)", gmean(en0["AUTO"]), gmean(en0["HILO"]))]
summary0 = pd.DataFrame([{"Metric": m, "AUTO": a, "HILO": h, "Saving (%)": 100 * (1 - h / a),
                          "Improvement (x)": a / h} for m, a, h in rows])
display(summary0.style.format({"AUTO": "{:,.2f}", "HILO": "{:,.2f}", "Saving (%)": "{:.2f}",
                               "Improvement (x)": "{:.4f}"}).hide(axis="index"))
summary0.to_csv(f"hilo_vs_auto_overall_noPP_{ROW}x{XB_ROWS}.csv", index=False)

# ---------------- bar chart ----------------
labels = [b[0] for b in BENCH] + ["GMean"]
data = {"Norm. latency": {"AUTO": [1.0] * 16, "HILO": nl + [gmean(nl)]},
        "Norm. energy": {"AUTO": [1.0] * 16, "HILO": ne + [gmean(ne)]}}
x = np.arange(len(labels), dtype=float); x[-1] += 0.6
w = 0.36
sty = {"AUTO": ("#1D9E75", "xxx"), "HILO": ("#EF9F27", "///")}
fig, axs = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for ax, (lab, d) in zip(axs, data.items()):
    for j, t in enumerate(d):
        colr, hat = sty[t]
        ax.bar(x + (j - 0.5) * w, d[t], w, label=t, color=colr, hatch=hat,
               edgecolor="black", linewidth=0.4)
    ax.axhline(1.0, color="black", linewidth=0.6, linestyle="--")
    ax.set_ylim(0.90, 1.01); ax.set_yticks(np.arange(0.90, 1.0001, 0.02))
    ax.set_ylabel(lab, fontweight="bold"); ax.grid(axis="y", alpha=0.3)
axs[1].set_xticks(x); axs[1].set_xticklabels(labels, rotation=40, ha="right")
axs[1].get_xticklabels()[-1].set_fontweight("bold")
axs[0].legend(ncol=2, loc="upper center", bbox_to_anchor=(0.5, 1.22), frameon=False)
fig.tight_layout()
fig.savefig(f"mvm_hilo_vs_auto_{ROW}x{XB_ROWS}_noPP.png", dpi=300)
fig.savefig(f"mvm_hilo_vs_auto_{ROW}x{XB_ROWS}_noPP.pdf")
plt.show()


In [ ]:
# Latency / energy of HILO vs AUTO using the FUNCTIONAL CELLS computed per adder
# Needs lookuptable_with_functional_cells.csv (in this folder).
#   cells per multiplication = inputs + pp segment + carry cells + output + max functional cells (computed)
#   each adder must fit: its functional cells <= free cells of the row (else the row is too small)
#   SET batches per adder: 1 if its gates fit in the free cells, else free cells are re-SET and reused
import math, numpy as np, pandas as pd, matplotlib.pyplot as plt

LUT = pd.read_csv("lookuptable_with_functional_cells.csv")
FC = {}
for _, r in LUT.iterrows():
    v = [int(r[f"col{i}"]) for i in range(1, 9)]
    while v and v[-1] == 0:
        v.pop()
    if pd.notna(r["functional_cells"]):
        FC[(tuple(v), int(r["weight"]))] = int(r["functional_cells"])


def multiply_fc(G, pp_ops):
    steps, carry_peak = schedule(G)
    fixed = 2 * N + max(s["inputs"] for s in steps) + carry_peak + 2 * N
    missing = [k for k, w, _, _ in G if (tuple(k), w) not in FC]
    if missing:
        raise KeyError(f"adders not in the functional-cells table: {missing[:5]}")
    need = max(FC[(tuple(k), w)] for k, w, _, _ in G)
    pool = ROW - fixed - (2 if pp_ops == 3 else 0)
    if pool < need:
        raise ValueError(f"row of {ROW} cells too small: needs {fixed + need}")
    ops = dict(gates=0, copy=0, pp=0, set_pp=0, set_func=0)
    for (k, w, cost, col), s in zip(G, steps):
        ops["gates"] += cost
        ops["copy"] += COPY_GATES * s["copied"]
        ops["pp"] += pp_ops * s["new_pp"]
        ops["set_pp"] += 1
        ops["set_func"] += max(1, math.ceil(cost / pool))
    ops["pp"] += pp_ops                                   # a0*b0
    evals = ops["gates"] + ops["copy"] + ops["pp"]
    return dict(cycles=evals + ops["set_pp"] + ops["set_func"] + 1, energy=evals * (E_NOR + E_SET),
                fixed=fixed, functional=need, cells=fixed + need, pool=pool, ops=ops)


def evaluate(pp_ops):
    mul = {"AUTO": multiply_fc(AUTO, pp_ops), "HILO": multiply_fc(HILO, pp_ops)}
    L = {t: [] for t in mul}; E = {t: [] for t in mul}
    for name, n, nnz in BENCH:
        xb = math.ceil(nnz / XB_ROWS)
        for t in mul:
            c, _, ph = mvm2(n, nnz, mul[t], ADD[t])
            L[t].append(c * T_CYCLE_NS / 1e3)
            E[t].append(sum(phase_energy(ph, xb).values()) * T_CYCLE_NS / 1e6)
    return mul, L, E


gmean = lambda v: float(np.exp(np.mean(np.log(v))))
print(f"Array {ROW}x{XB_ROWS}, cycle {T_CYCLE_NS} ns\n")
per_mul, overall, keep = [], [], {}
for pp_ops, label in ((3, "3n^2"), (0, "none")):
    mul, L, E = evaluate(pp_ops); keep[label] = (L, E)
    for t, m in mul.items():
        per_mul.append({"PP generation": label, "tech": t, "functional cells (max)": m["functional"],
                        "cells per multiplication": m["cells"], "adder gates": m["ops"]["gates"],
                        "carry-copy NOTs": m["ops"]["copy"], "pp ops": m["ops"]["pp"],
                        "SET batches": m["ops"]["set_pp"] + m["ops"]["set_func"] + 1,
                        "cycles per multiply": m["cycles"]})
    for metric, a, h in (("Multiply cycles", mul["AUTO"]["cycles"], mul["HILO"]["cycles"]),
                         ("Latency, total of 15 MVMs (us)", sum(L["AUTO"]), sum(L["HILO"])),
                         ("Energy, total of 15 MVMs (uJ)", sum(E["AUTO"]), sum(E["HILO"])),
                         ("Latency, geometric mean (us)", gmean(L["AUTO"]), gmean(L["HILO"])),
                         ("Energy, geometric mean (uJ)", gmean(E["AUTO"]), gmean(E["HILO"]))):
        overall.append({"PP generation": label, "Metric": metric, "AUTO": a, "HILO": h,
                        "Saving (%)": 100 * (1 - h / a), "Improvement (x)": a / h})
display(pd.DataFrame(per_mul).style.hide(axis="index"))
OVR = pd.DataFrame(overall)
display(OVR.style.format({"AUTO": "{:,.2f}", "HILO": "{:,.2f}", "Saving (%)": "{:.2f}",
                          "Improvement (x)": "{:.4f}"}).hide(axis="index"))
OVR.to_csv(f"hilo_vs_auto_overall_functional_{ROW}x{XB_ROWS}.csv", index=False)

CHART_PP = "3n^2"                      # which case to draw: "3n^2" or "none"
L, E = keep[CHART_PP]
labels = [b[0] for b in BENCH] + ["GMean"]
nl = [h / a for h, a in zip(L["HILO"], L["AUTO"])]; ne = [h / a for h, a in zip(E["HILO"], E["AUTO"])]
data = {"Norm. latency": nl + [gmean(nl)], "Norm. energy": ne + [gmean(ne)]}
x = np.arange(len(labels), dtype=float); x[-1] += 0.6; w = 0.36
fig, axs = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for ax, (lab, hv) in zip(axs, data.items()):
    ax.bar(x - w / 2, [1.0] * len(hv), w, label="AUTO", color="#1D9E75", hatch="xxx",
           edgecolor="black", linewidth=0.4)
    ax.bar(x + w / 2, hv, w, label="HILO", color="#EF9F27", hatch="///", edgecolor="black", linewidth=0.4)
    ax.axhline(1.0, color="black", linewidth=0.6, linestyle="--")
    ax.set_ylim(0.90, 1.01); ax.set_yticks(np.arange(0.90, 1.0001, 0.02))
    ax.set_ylabel(lab, fontweight="bold"); ax.grid(axis="y", alpha=0.3)
axs[1].set_xticks(x); axs[1].set_xticklabels(labels, rotation=40, ha="right")
axs[1].get_xticklabels()[-1].set_fontweight("bold")
axs[0].legend(ncol=2, loc="upper center", bbox_to_anchor=(0.5, 1.22), frameon=False)
fig.tight_layout()
tag = f"{ROW}x{XB_ROWS}_functional_pp{'3n2' if CHART_PP == '3n^2' else 'none'}"
fig.savefig(f"mvm_hilo_vs_auto_{tag}.png", dpi=300); fig.savefig(f"mvm_hilo_vs_auto_{tag}.pdf")
plt.show()


In [ ]:
# Same functional-cell latency / energy evaluation for a 256 x 256 array
# (addition costs ADD do not depend on the array size, so no new search is needed)
ROW, XB_ROWS = 256, 256
src = [c for c in In if c.startswith('# Latency / energy of HILO vs AUTO using the FUNCTIONAL')][-1]
exec(src)

In [ ]:
# 256 x 256, WITHOUT partial-product generation: tables + bar chart
ROW, XB_ROWS = 256, 256
src = [c for c in In if c.startswith('# Latency / energy of HILO vs AUTO using the FUNCTIONAL')][-1]
src = src.replace('CHART_PP = "3n^2"', 'CHART_PP = "none"')
# grayscale: AUTO light gray, HILO dark gray (hatches kept so they also differ in black-and-white print)
src = src.replace('color="#1D9E75"', 'color="#d9d9d9"').replace('color="#EF9F27"', 'color="#6b6b6b"')
src = src.replace('_functional_pp', '_functional_gray_pp')
exec(src)
print('Without partial products:')
display(OVR[OVR['PP generation'] == 'none'].style.format({'AUTO': '{:,.2f}', 'HILO': '{:,.2f}', 'Saving (%)': '{:.2f}', 'Improvement (x)': '{:.4f}'}).hide(axis='index'))

In [ ]:
# LOGIC vs AUTO vs HILO, normalized to AUTO, with HILO's improvement over LOGIC and AUTO
# LOGIC (TODAES'25, Sec. 2.2.2 / 4.1): an n-bit multiply = n-1 sequential ROW additions,
#   IS_k = IS_(k-1) + PP_k, each n-bit addition = a chain of 8-bit adders; 10,046 timesteps in total.
# Modelled with the same row-mapping rules as AUTO / HILO (one multiplication per row):
#   each row addition = one adder step: new partial-product row PP_k (32 bits, made on demand) + IS
#   the published 10,046 timesteps are used as the adder gates (they already include LOGIC's own moves)
#   functional cells = LOGIC's reported intermediate storage of a 32-bit addition (42, Table 1)
#   row = inputs 64 + PP row 32 + IS window 33 + output 64 + functional 42
import math, numpy as np, pandas as pd, matplotlib.pyplot as plt

PP_CASE = "none"            # "none" = without partial-product generation, "3n^2" = with
LOGIC_FUNC = 42             # LOGIC Table 1, intermediate storage of a 32-bit addition
LOGIC_COPY = False          # True = also copy IS (32 bits, NOT+NOT) between row additions
LOGIC_GATES = 10046         # LOGIC Table 1, 32-bit multiplication (timesteps)
LOGIC_ADD32 = 322           # LOGIC Table 1, 32-bit addition (timesteps)
ROW_ADDS, CHUNK = N - 1, 8  # 31 row additions, 8-bit adder chain (LOGIC Sec. 7.2)


def multiply_logic(pp_ops):
    f = LOGIC_FUNC
    seg = dict(inputs=2 * N, pp_row=N, IS=N + 1, output=2 * N)
    fixed = sum(seg.values())
    pool = ROW - fixed - (2 if pp_ops == 3 else 0)
    if pool < f:
        raise ValueError(f"LOGIC needs {fixed + f} cells, row has {ROW}")
    per = [LOGIC_GATES // ROW_ADDS + (1 if i < LOGIC_GATES % ROW_ADDS else 0) for i in range(ROW_ADDS)]
    ops = dict(gates=sum(per), copy=COPY_GATES * N * (ROW_ADDS - 1) if LOGIC_COPY else 0, pp=pp_ops * N * N,
               set_pp=ROW_ADDS, set_func=sum(max(1, math.ceil(c / pool)) for c in per))
    evals = ops["gates"] + ops["copy"] + ops["pp"]
    return dict(cycles=evals + ops["set_pp"] + ops["set_func"] + 1, energy=evals * (E_NOR + E_SET),
                fixed=fixed, functional=f, cells=fixed + f, pool=pool, ops=ops)


# LOGIC adds k numbers as k-1 sequential 32-bit additions (4 chained 8-bit adders each)
ADD_LOGIC = {k: dict(gates=(k - 1) * LOGIC_ADD32, cycles=(k - 1) * (LOGIC_ADD32 + N // CHUNK))
             for k in ADD["AUTO"]}

pp_ops = 3 if PP_CASE == "3n^2" else 0
MULS = {"LOGIC": multiply_logic(pp_ops), "AUTO": multiply_fc(AUTO, pp_ops), "HILO": multiply_fc(HILO, pp_ops)}
ADDS = {"LOGIC": ADD_LOGIC, "AUTO": ADD["AUTO"], "HILO": ADD["HILO"]}
TECHS = list(MULS)

L = {t: [] for t in TECHS}; E = {t: [] for t in TECHS}
for name, n, nnz in BENCH:
    xb = math.ceil(nnz / XB_ROWS)
    for t in TECHS:
        c, _, ph = mvm2(n, nnz, MULS[t], ADDS[t])
        L[t].append(c * T_CYCLE_NS / 1e3)                                   # us
        E[t].append(sum(phase_energy(ph, xb).values()) * T_CYCLE_NS / 1e6)  # uJ
gm = lambda v: float(np.exp(np.mean(np.log(v))))

print(f"Array {ROW}x{XB_ROWS}, partial products: {PP_CASE}, cycle {T_CYCLE_NS} ns\n")
display(pd.DataFrame([{"tech": t, "cells per multiplication": m["cells"], "functional cells": m["functional"],
                       "adder gates": m["ops"]["gates"], "copy NOTs": m["ops"]["copy"],
                       "pp ops": m["ops"]["pp"], "cycles per multiply": m["cycles"]}
                      for t, m in MULS.items()]).style.hide(axis="index"))

# ---------- overall values and HILO's improvement ----------
metrics = [("Multiply cycles", {t: MULS[t]["cycles"] for t in TECHS}),
           ("Cells per multiplication", {t: MULS[t]["cells"] for t in TECHS}),
           ("Latency, total of 15 MVMs (us)", {t: sum(L[t]) for t in TECHS}),
           ("Energy, total of 15 MVMs (uJ)", {t: sum(E[t]) for t in TECHS}),
           ("Latency, geometric mean (us)", {t: gm(L[t]) for t in TECHS}),
           ("Energy, geometric mean (uJ)", {t: gm(E[t]) for t in TECHS})]
rows = []
for name, v in metrics:
    rows.append({"Metric": name, "LOGIC": v["LOGIC"], "AUTO": v["AUTO"], "HILO": v["HILO"],
                 "HILO vs LOGIC (%)": 100 * (1 - v["HILO"] / v["LOGIC"]), "HILO vs LOGIC (x)": v["LOGIC"] / v["HILO"],
                 "HILO vs AUTO (%)": 100 * (1 - v["HILO"] / v["AUTO"]), "HILO vs AUTO (x)": v["AUTO"] / v["HILO"]})
CMP = pd.DataFrame(rows)
display(CMP.style.format({"LOGIC": "{:,.2f}", "AUTO": "{:,.2f}", "HILO": "{:,.2f}",
                          "HILO vs LOGIC (%)": "{:.2f}", "HILO vs LOGIC (x)": "{:.4f}",
                          "HILO vs AUTO (%)": "{:.2f}", "HILO vs AUTO (x)": "{:.4f}"}).hide(axis="index"))
tag = f"{ROW}x{XB_ROWS}_pp{'3n2' if pp_ops else 'none'}"
CMP.to_csv(f"logic_auto_hilo_overall_{tag}.csv", index=False)

# ---------- Times New Roman for the figure ----------
import glob, subprocess, matplotlib as mpl
from matplotlib import font_manager as fm
def _tnr_files():
    return [f for f in glob.glob("/usr/share/fonts/**/*.ttf", recursive=True) if "times" in f.lower()]
if not _tnr_files():   # install Microsoft core fonts (Times New Roman) once
    subprocess.run("echo ttf-mscorefonts-installer msttcorefonts/accepted-mscorefonts-eula select true | "
                   "debconf-set-selections && apt-get -qq install -y ttf-mscorefonts-installer",
                   shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for f in _tnr_files():
    fm.fontManager.addfont(f)
have_tnr = any(f.name == "Times New Roman" for f in fm.fontManager.ttflist)
mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = ["Times New Roman", "Liberation Serif", "Nimbus Roman", "DejaVu Serif"]
mpl.rcParams["mathtext.fontset"] = "stix"
mpl.rcParams["font.weight"] = "bold"          # all text bold: ticks, labels, legend
mpl.rcParams["axes.labelweight"] = "bold"
mpl.rcParams["mathtext.default"] = "bf"
print("Font:", "Times New Roman" if have_tnr else "Times New Roman not found, using Liberation Serif (same metrics)")

# ---------- bar chart, normalized to AUTO, grayscale ----------
labels = [b[0] for b in BENCH]          # 15 benchmarks only (no GMean bar)
norm = {}
for key, d in (("lat", L), ("en", E)):
    for t in TECHS:
        r = [x / a for x, a in zip(d[t], d["AUTO"])]
        norm[(key, t)] = r
sty = {"LOGIC": ("#ffffff", "\\\\\\"), "AUTO": ("#d9d9d9", "xxx"), "HILO": ("#6b6b6b", "///")}
x = np.arange(len(labels), dtype=float)
w = 0.27
fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
for ax, key, lab in [(axs[0], "lat", "Norm. latency"), (axs[1], "en", "Norm. energy")]:
    for j, t in enumerate(TECHS):
        colr, hat = sty[t]
        ax.bar(x + (j - 1) * w, norm[(key, t)], w, label=t, color=colr, hatch=hat,
               edgecolor="black", linewidth=0.5)
    ax.axhline(1.0, color="black", linewidth=0.6, linestyle="--")
    top = max(max(norm[(key, t)]) for t in TECHS)
    ax.set_ylim(0.90, top + 0.005)                   # tight top, no white gap
    ax.set_ylabel(lab, fontweight="bold", fontsize=15); ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="y", labelsize=13)
axs[1].set_xticks(x); axs[1].set_xticklabels(labels, rotation=40, ha="right", fontsize=13)
axs[0].legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.36), frameon=False, prop={"weight": "bold", "size": 15},
              handlelength=4.5, handleheight=2.2, handletextpad=0.6, columnspacing=2.5)
fig.tight_layout()
fig.savefig(f"mvm_logic_auto_hilo_{tag}_gray.png", dpi=300)
fig.savefig(f"mvm_logic_auto_hilo_{tag}_gray.pdf")
plt.show()


In [ ]:
# Same LOGIC / AUTO / HILO results, normalized to LOGIC (run the LOGIC vs AUTO vs HILO cell first)
import math, numpy as np, pandas as pd, matplotlib as mpl, matplotlib.pyplot as plt

mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = ["Times New Roman", "Liberation Serif", "Nimbus Roman", "DejaVu Serif"]
mpl.rcParams["mathtext.fontset"] = "stix"
mpl.rcParams["font.weight"] = "bold"
mpl.rcParams["axes.labelweight"] = "bold"
mpl.rcParams["mathtext.default"] = "bf"

# ---------- overall values: HILO vs LOGIC and HILO vs AUTO ----------
rows = []
for metric, d in (("Latency, total of 15 MVMs (us)", {t: sum(L[t]) for t in TECHS}),
                  ("Energy, total of 15 MVMs (uJ)", {t: sum(E[t]) for t in TECHS}),
                  ("Latency, geometric mean (us)", {t: gm(L[t]) for t in TECHS}),
                  ("Energy, geometric mean (uJ)", {t: gm(E[t]) for t in TECHS})):
    rows.append({"Metric": metric, "LOGIC": d["LOGIC"], "AUTO": d["AUTO"], "HILO": d["HILO"],
                 "HILO vs LOGIC (%)": 100 * (1 - d["HILO"] / d["LOGIC"]), "HILO vs LOGIC (x)": d["LOGIC"] / d["HILO"],
                 "HILO vs AUTO (%)": 100 * (1 - d["HILO"] / d["AUTO"]), "HILO vs AUTO (x)": d["AUTO"] / d["HILO"]})
NL = pd.DataFrame(rows)
display(NL.style.format({c: "{:,.2f}" for c in ("LOGIC", "AUTO", "HILO", "HILO vs LOGIC (%)", "HILO vs AUTO (%)")}
                        | {c: "{:.4f}" for c in ("HILO vs LOGIC (x)", "HILO vs AUTO (x)")}).hide(axis="index"))
tag = f"{ROW}x{XB_ROWS}_pp{'3n2' if pp_ops else 'none'}"
NL.to_csv(f"logic_auto_hilo_normLOGIC_{tag}.csv", index=False)

# ---------- bar chart, normalized to LOGIC, grayscale, 15 benchmarks ----------
labels = [b[0] for b in BENCH]
norm = {(key, t): [x / g for x, g in zip(d[t], d["LOGIC"])] for key, d in (("lat", L), ("en", E)) for t in TECHS}
sty = {"LOGIC": ("#ffffff", "\\\\\\"), "AUTO": ("#d9d9d9", "xxx"), "HILO": ("#6b6b6b", "///")}
x = np.arange(len(labels), dtype=float)
w = 0.27
fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
for ax, key, lab in [(axs[0], "lat", "Norm. latency"), (axs[1], "en", "Norm. energy")]:
    for j, t in enumerate(TECHS):
        colr, hat = sty[t]
        ax.bar(x + (j - 1) * w, norm[(key, t)], w, label=t, color=colr, hatch=hat,
               edgecolor="black", linewidth=0.5)
    ax.axhline(1.0, color="black", linewidth=0.6, linestyle="--")
    low = min(min(norm[(key, t)]) for t in TECHS)
    ax.set_ylim(math.floor((low - 0.02) * 20) / 20, 1.005)  # bottom below HILO, tight top (LOGIC = 1)
    ax.set_ylabel(lab, fontweight="bold", fontsize=15); ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="y", labelsize=13)
axs[1].set_xticks(x); axs[1].set_xticklabels(labels, rotation=40, ha="right", fontsize=13)
axs[0].legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.36), frameon=False, prop={"weight": "bold", "size": 15},
              handlelength=4.5, handleheight=2.2, handletextpad=0.6, columnspacing=2.5)
fig.tight_layout()
fig.savefig(f"mvm_logic_auto_hilo_normLOGIC_{tag}_gray.png", dpi=300)
fig.savefig(f"mvm_logic_auto_hilo_normLOGIC_{tag}_gray.pdf")
plt.show()


In [ ]:
# ULTRA [Imani, DAC'17] and SIMPLER [Haj-Ali, ISCAS'18] added to LOGIC / AUTO / HILO, normalized to ULTRA
# (run the "LOGIC vs AUTO vs HILO" cell first: it sets MULS, ADDS, pp_ops, mvm2, phase_energy, ...)
# Published 32-bit multiplication (Haj-Ali Table II, cycle-accurate):
#   ULTRA      15,007 cycles, 15,071 memristors (15N^2-11N-1, 15N^2-9N-1)
#   SIMPLER  12,870 cycles,    635 memristors (full precision, 13N^2-14N+6, 20N-5)
# Both use the 12N+1-cycle MAGIC adder; ULTRA also has a 3:2 (CSA) step of 13 cycles.
# Same row-mapping rules as before: a multiplication needs its cells in the array; if it needs more cells
# than one row (ROW), it is folded over ceil(cells / ROW) rows (no extra cycles charged for folding), so a
# crossbar holds only XB_ROWS // rows_per_mul multiplications at once -> more crossbars for the same nnz.
import math, numpy as np, pandas as pd, matplotlib as mpl, matplotlib.pyplot as plt

mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = ["Times New Roman", "Liberation Serif", "Nimbus Roman", "DejaVu Serif"]
mpl.rcParams["mathtext.fontset"] = "stix"
mpl.rcParams["font.weight"] = "bold"
mpl.rcParams["axes.labelweight"] = "bold"
mpl.rcParams["mathtext.default"] = "bf"

ADD_N = 12 * N + 1                                     # MAGIC N-bit adder used by both papers
PUB = {"ULTRA": dict(cycles=15007, cells=15071), "SIMPLER": dict(cycles=12870, cells=20 * N - 5)}
ADD_PUB = {"ULTRA": {k: dict(gates=ADD_N + 13 * (k - 2), cycles=ADD_N + 13 * (k - 2)) for k in ADDS["AUTO"]},
           "SIMPLER": {k: dict(gates=(k - 1) * ADD_N, cycles=(k - 1) * ADD_N) for k in ADDS["AUTO"]}}


def mvm_slots(n, nnz, mul, add, slots):
    """mvm2 with `slots` multiplications per crossbar (slots = XB_ROWS when one multiply fits one row)."""
    q, r = divmod(nnz, n)
    lens = [q + 1] * r + [q] * (n - r)
    xbars, cur, fill, split = [], [], 0, 0
    for Lr in lens:
        while Lr:
            take = min(Lr, slots - fill)
            cur.append(take); fill += take; Lr -= take
            if Lr:
                split += 1
            if fill == slots:
                xbars.append(cur); cur, fill = [], 0
    if cur:
        xbars.append(cur)
    t = {"load": bus(n * N) + XB_ROWS, "multiply": mul["cycles"]}
    t["reduce"] = max(reduce_cost(s, add)[0] for s in xbars)
    t["merge"] = (bus(split * ACC_BITS) + add[2]["cycles"]) if split else 0
    t["readout"] = XB_ROWS + bus(n * ACC_BITS)
    return sum(t.values()), t, len(xbars)


ALLM = {"ULTRA": dict(cycles=PUB["ULTRA"]["cycles"], cells=PUB["ULTRA"]["cells"]),
        "SIMPLER": dict(cycles=PUB["SIMPLER"]["cycles"], cells=PUB["SIMPLER"]["cells"])}
ALLM.update({t: dict(cycles=MULS[t]["cycles"], cells=MULS[t]["cells"]) for t in ("LOGIC", "AUTO", "HILO")})
ALLA = {"ULTRA": ADD_PUB["ULTRA"], "SIMPLER": ADD_PUB["SIMPLER"], **{t: ADDS[t] for t in ("LOGIC", "AUTO", "HILO")}}
T5 = list(ALLM)
for t, m in ALLM.items():
    m["rows_per_mul"] = math.ceil(m["cells"] / ROW)
    m["mul_per_xbar"] = XB_ROWS // m["rows_per_mul"]

L5 = {t: [] for t in T5}; E5 = {t: [] for t in T5}; XB5 = {t: [] for t in T5}
for name, n, nnz in BENCH:
    for t in T5:
        c, ph, xb = mvm_slots(n, nnz, ALLM[t], ALLA[t], ALLM[t]["mul_per_xbar"])
        L5[t].append(c * T_CYCLE_NS / 1e3)                                   # us
        E5[t].append(sum(phase_energy(ph, xb).values()) * T_CYCLE_NS / 1e6)  # uJ
        XB5[t].append(xb)

print(f"Array {ROW}x{XB_ROWS}, partial products for LOGIC/AUTO/HILO: {PP_CASE}, cycle {T_CYCLE_NS} ns\n")
display(pd.DataFrame([{"tech": t, "cycles per multiply": m["cycles"], "cells per multiplication": m["cells"],
                       "rows per multiplication": m["rows_per_mul"], "multiplications per crossbar": m["mul_per_xbar"],
                       "crossbars (sum of 15)": sum(XB5[t])} for t, m in ALLM.items()]).style.hide(axis="index"))

# ---------- overall values: HILO vs every technique ----------
rows = []
for metric, d in (("Multiply cycles", {t: ALLM[t]["cycles"] for t in T5}),
                  ("Latency, total of 15 MVMs (us)", {t: sum(L5[t]) for t in T5}),
                  ("Energy, total of 15 MVMs (uJ)", {t: sum(E5[t]) for t in T5}),
                  ("Latency, geometric mean (us)", {t: gm(L5[t]) for t in T5}),
                  ("Energy, geometric mean (uJ)", {t: gm(E5[t]) for t in T5})):
    row = {"Metric": metric, **{t: d[t] for t in T5}}
    for b in ("ULTRA", "SIMPLER", "LOGIC", "AUTO"):
        row[f"HILO vs {b} (%)"] = 100 * (1 - d["HILO"] / d[b])
        row[f"HILO vs {b} (x)"] = d[b] / d["HILO"]
    rows.append(row)
U5 = pd.DataFrame(rows)
display(U5.style.format({c: ("{:.4f}" if c.endswith("(x)") else "{:,.2f}") for c in U5.columns if c != "Metric"})
        .hide(axis="index"))
tag = f"{ROW}x{XB_ROWS}_pp{'3n2' if pp_ops else 'none'}"
U5.to_csv(f"ultra_simpler_logic_auto_hilo_normULTRA_{tag}.csv", index=False)

# ---------- bar chart, normalized to ULTRA, grayscale, 15 benchmarks ----------
labels = [b[0] for b in BENCH]
norm = {(key, t): [v / u for v, u in zip(d[t], d["ULTRA"])] for key, d in (("lat", L5), ("en", E5)) for t in T5}
sty = {"ULTRA": ("#ffffff", "---"), "SIMPLER": ("#ffffff", "..."), "LOGIC": ("#ffffff", "\\\\\\"),
       "AUTO": ("#d9d9d9", "xxx"), "HILO": ("#6b6b6b", "///")}
x = np.arange(len(labels), dtype=float)
w = 0.17
fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
for ax, key, lab in [(axs[0], "lat", "Norm. latency"), (axs[1], "en", "Norm. energy (log)")]:
    for j, t in enumerate(T5):
        colr, hat = sty[t]
        ax.bar(x + (j - 2) * w, norm[(key, t)], w, label=t, color=colr, hatch=hat,
               edgecolor="black", linewidth=0.5)
    ax.axhline(1.0, color="black", linewidth=0.6, linestyle="--")
    if key == "en":            # ULTRA uses ~64x more crossbars -> energy spans 3 decades, log axis
        low = min(min(norm[(key, t)]) for t in T5)
        ax.set_yscale("log"); ax.set_ylim(10 ** math.floor(math.log10(low)), 1.3)
        ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:g}"))
    else:                      # start just below the lowest bar, not at zero
        low = min(min(norm[(key, t)]) for t in T5)
        ax.set_ylim(math.floor((low - 0.05) * 10) / 10, 1.02)
    ax.set_ylabel(lab, fontweight="bold", fontsize=15); ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="y", labelsize=13)
axs[1].set_xlim(x[0] - 0.55, x[-1] + 0.55)   # no empty space at the ends
axs[1].set_xticks(x); axs[1].set_xticklabels(labels, rotation=40, ha="right", fontsize=13)
axs[0].legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, 1.36), frameon=False, prop={"weight": "bold", "size": 15},
              handlelength=3.5, handleheight=2.2, handletextpad=0.5, columnspacing=1.8)
fig.tight_layout()
fig.savefig(f"mvm_ultra_simpler_logic_auto_hilo_normULTRA_{tag}_gray.png", dpi=300)
fig.savefig(f"mvm_ultra_simpler_logic_auto_hilo_normULTRA_{tag}_gray.pdf")
plt.show()
